# A dynamic hydrocracker, and the reformer's recycle loop

`difflow_refinery.dynamic.DynamicHydrocracker` runs a solved steady `Hydrocracker` in time: its pretreat and cracking beds as cells with gas, liquid and catalyst holdup, the cracking quench valves as integral controllers on each bed's inlet temperature (what the steady unit's solved quench stands for), the HP separator's level, the recycle loop's gas inventory, and the fractionator behind product holdups -- whose UCO holdup carries the UCO recycle, so neither of the steady unit's tears is solved. Its rest state is the steady unit's answer (see the gates in `docs/unit-operations-refinery.md`, "Dynamic operation").

This notebook: build it on a straight-run VGO and settle it; run a feed-nitrogen excursion (the pretreat bed's job is to protect the cracking catalyst from it); take a gradient through that trajectory; show what happens with a setpoint step at the default holdups (it is too stiff for the explicit integrator -- an open problem, reported rather than hidden); then the catalytic reformer with its dynamic separator and recycle loop through a partial compressor trip.

Every holdup and time constant is ILLUSTRATIVE (`HydrocrackerHoldupParams`): the transients have not been compared with a plant or a published dynamic model. Run time: about 40 minutes on a laptop CPU.

In [ ]:
import time, warnings
import jax, jax.numpy as jnp, numpy as np
import matplotlib.pyplot as plt
jax.config.update("jax_enable_x64", True)

import difflow_refinery as dr
from difflow_refinery.hydrocracking.unit import Hydrocracker, HydrocrackerParams
from difflow_refinery.hydrotreating.feed import select_cuts, straight_run_cut
from difflow_refinery.dynamic import DynamicHydrocracker, DynamicRunawayWarning, DynamicReformerTrain

def vgo(sg=0.86, sulfur=1.8, nitrogen=1500.0):
    a = dr.Assay([5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95],
                 jnp.asarray([t + 273.15 for t in (60, 95, 150, 205, 260, 315, 370, 430, 500, 600, 680)]),
                 sg=sg, light_ends={"propane": 0.5, "n_butane": 1.0, "n_pentane": 1.5},
                 heavy_end=dr.HeavyEnd(), sulfur_wt=sulfur, nitrogen_wppm=nitrogen)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        char = dr.characterize(a, composition=True)
    return char, straight_run_cut(char, rate=50.0, cuts=select_cuts(char, 370 + 273.15, 560 + 273.15))

## 1. The steady unit and its dynamic twin

A 370-560 C straight-run VGO of the docs' light crude, 50 kg/s, once through, at the default specs (cracking inlets 380 C, quench solved for them).

In [ ]:
char, feed = vgo()
unit = Hydrocracker(char, feed, HydrocrackerParams())
t0 = time.time()
steady = unit.solve(feed)
print(f"steady solve {time.time() - t0:.0f} s (mostly compile)")
print(steady.table())

t0 = time.time()
dyn = DynamicHydrocracker.from_steady(unit, feed, steady)          # 5 cells per bed
y_rest = dyn.settle(4 * 3600.0)
print(f"\n{dyn.y0.size} states; built and settled for 4 h in {time.time() - t0:.0f} s")
ob, ref = dyn.observe(0.0, y_rest), dyn.steady_reference()
for k in ("conversion.per_pass", "wabt.pretreat", "wabt.crack", "h2.makeup", "recycle.rate"):
    print(f"{k:22s} dynamic {float(ob[k]):12.5f}   steady {float(ref[k]):12.5f}")
print("quench valves (fraction of the treat gas):", np.round(np.asarray(ob["crack.quench"]), 5),
      " steady:", np.round(np.asarray(ref["crack.quench"]), 5))

## 2. A feed-nitrogen excursion

The feed's organic nitrogen goes up 50 % at one minute (`feed_nitrogen`, a multiplier). The pretreat bed removes most of it; what slips through inhibits the cracking rates.

In [ ]:
a = dyn.args(steps={"feed_nitrogen": [(60.0, 0.5)]})
t0 = time.time()
sim = dyn.simulate(3600.0, args=a, y0=y_rest, n_save=61)
tr = dyn.trajectory(sim, a, ["conversion.per_pass", "pretreat.N_wppm", "wabt.crack", "crack.quench"])
print(f"{time.time() - t0:.0f} s, {int(sim.info.n_steps)} steps, completed {bool(sim.info.success)}")
t = np.asarray(tr["t"]) / 60
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
ax[0].plot(t, np.asarray(tr["pretreat.N_wppm"])); ax[0].set_ylabel("organic N to the cracker (wppm)")
ax[1].plot(t, 100 * np.asarray(tr["conversion.per_pass"])); ax[1].set_ylabel("conversion per pass (%)")
ax[2].plot(t, np.asarray(tr["crack.quench"])); ax[2].set_ylabel("quench valves (fraction of treat gas)")
for a_ in ax: a_.set_xlabel("time (min)")
fig.tight_layout()

## 3. A gradient through the trajectory

d(organic N to the cracker 15 minutes into the excursion)/d(size of the nitrogen step), by reverse mode through the diffrax solve, against central differences.

In [ ]:
def n_at_15min(dN):
    a = dyn.args(steps={"feed_nitrogen": [(60.0, dN)]})
    s = dyn.simulate(900.0, args=a, y0=y_rest, n_save=2, warn=False)
    return dyn.observe(s.trajectory.t[-1], s.y_final, a)["pretreat.N_wppm"]

t0 = time.time()
g = float(jax.grad(n_at_15min)(jnp.asarray(0.5)))
print(f"reverse mode: {g:.6e} wppm per unit multiplier ({time.time() - t0:.0f} s with compile)")
for h in (0.1, 0.05):
    fd = (float(n_at_15min(jnp.asarray(0.5 + h))) - float(n_at_15min(jnp.asarray(0.5 - h)))) / (2 * h)
    print(f"central difference h = {h}: {fd:.6e}")

## 4. What is still stiff

A step of the cracking setpoint (or a recycle-compressor trip) is a much larger disturbance: the quench controllers, the beds' gas holdup and the catalyst's heat interact, and the explicit integrator (Tsit5) cannot take large steps. Capped at 3000 steps, the run stops early and says so -- `simulate` returns with `info.success` False and a `DynamicRunawayWarning`, rather than raising or running for hours. An implicit (stiff) integrator with a sparse Jacobian is the fix this needs; it is an open problem.

In [ ]:
a = dyn.args(steps={"T_crack": [(300.0, 5.0)]})
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always", DynamicRunawayWarning)
    sim = dyn.simulate(3600.0, args=a, y0=y_rest, n_save=13, max_steps=3000)
for x in w:
    print("warning:", x.message)
print(dyn.excursion(sim, a))

## 5. The reformer with its recycle loop

`DynamicReformerTrain.from_steady(..., recycle_loop=True)` makes the separator and the recycle loop dynamic (`DynamicReformerLoop`): the separator liquid holdup, the loop's gas inventory, the compressor on the H2/HC spec and the net gas on a pressure controller. Its rest state is the steady reformer's. A partial compressor trip (to 60 % for ten minutes) cuts the hydrogen to the reactors.

In [ ]:
from difflow_refinery.reforming import CatalyticReformer, ReformerParams, lean_naphtha
naphtha = lean_naphtha()
reformer = CatalyticReformer(ReformerParams())
steady_r = reformer.solve(naphtha)
tr = DynamicReformerTrain.from_steady(reformer, naphtha, steady_r, recycle_loop=True)
t0 = time.time()
yr = tr.y0
a = tr.args(steps={"compressor": [(300.0, -0.4), (900.0, 0.4)]}, ramp=30.0)
sim = tr.simulate(2400.0, args=a, y0=yr, n_save=81)
o = tr.trajectory(sim, a, ["T_out", "reformate.RON", "recycle.rate", "h2.make"])
print(f"{time.time() - t0:.0f} s; RON at rest {float(o['reformate.RON'][0]):.3f} (steady {float(steady_r.outputs()['reformate.RON']):.3f})")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
t = np.asarray(o["t"]) / 60
ax[0].plot(t, np.asarray(o["recycle.rate"])); ax[0].set_ylabel("recycle (mol/s)")
ax[1].plot(t, np.asarray(o["T_out"]) - 273.15); ax[1].set_ylabel("reactor outlets (C)")
ax[2].plot(t, np.asarray(o["reformate.RON"])); ax[2].set_ylabel("reformate RON")
for a_ in ax: a_.set_xlabel("time (min)")
fig.tight_layout()

## What this is and is not

* The rest states are the steady units' (the hydrocracker's to the cell discretisation, the reformer loop's to round-off); element balances close at every state of a trajectory; gradients through trajectories match central differences (the numbers are in the docs' gates).
* Setpoint steps and compressor trips of the hydrocracker are too stiff for the explicit integrator at these holdups (section 4); so is a start far from rest. That is the main open problem.
* The holdups, catalyst heat capacity, controller time constants, valve capacities and the alarm temperature are illustrative. A cell's gas residence time is floored at 20 s (`min_gas_tau`) so the explicit integrator steps at the liquid's and the catalyst's pace, not the gas's; that slows the gas's transit through a bed and nothing else.
* Pressures are fixed. The fractionator is the steady TBP split behind product holdups, not a column with tray dynamics. No transient here has been compared with a plant.